In [ ]:
# ===============================
# 1. BASIC SETUP & LIBRARIES
# ===============================

import pandas as pd
import numpy as np
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/content/data cashflow.csv")

##Value Count

In [ ]:
df["uraian"].value_counts()

In [ ]:
df["tanggal"].value_counts()

In [ ]:
df["bulan"].value_counts()

In [ ]:
df["i/o"].value_counts()

In [ ]:
df["qty"].value_counts()

In [ ]:
df["amount"].value_counts()

##Standarisasi & Normalisasi

###Standarisasi i/o

In [ ]:
df["io_clean"] = df["i/o"].str.lower().str.strip()


###Standarisasi & Urutkan Bulan

In [ ]:
bulan_map = {
    "mei": 5,
    "juni": 6,
    "juli": 7,
    "agustus": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "desember": 12
}

df["bulan_clean"] = df["bulan"].str.lower().str.strip()
df["bulan_index"] = df["bulan_clean"].map(bulan_map)


In [ ]:
df[df["bulan_index"].isna()]["bulan"].unique()


###Kolom cashflow_amount

In [ ]:
df["cashflow_amount"] = np.where(
    df["io_clean"] == "in",
    df["amount"],
    -df["amount"]
)


###Uraian_clean

In [ ]:
df["uraian_clean"] = df["uraian"].str.lower().str.strip()


###Split QTY

In [ ]:
df["qty_str"] = df["qty"].astype(str)

df["qty_value"] = (
    df["qty_str"]
    .str.extract(r"(\d+)")
    .astype(float)
)

df["qty_unit"] = df["qty_str"].str.replace(r"\d+", "", regex=True).str.strip()


In [ ]:
df.sample(5)

#Analisa

##1. Cashflow Analysis

In [ ]:
df[["bulan_clean","bulan_index","io_clean","amount","cashflow_amount"]].head()


##Cashflow Aggregation
##Total pemasukan & penegluaran per bulan

In [ ]:
monthly_summary = (
    df
    .groupby(["bulan_index", "io_clean"])["amount"]
    .sum()
    .reset_index()
)


In [ ]:
monthly_pivot = (
    monthly_summary
    .pivot(index="bulan_index", columns="io_clean", values="amount")
    .fillna(0)
    .reset_index()
)

monthly_pivot.columns.name = None
monthly_pivot.rename(columns={
    "in": "total_pemasukan",
    "out": "total_pengeluaran"
}, inplace=True)


###Net Cash Flow per Bulan

In [ ]:
monthly_pivot["net_cash_flow"] = (
    monthly_pivot["total_pemasukan"]
    - monthly_pivot["total_pengeluaran"]
)


In [ ]:
bulan_reverse_map = {
    5: "Mei",
    6: "Juni",
    7: "Juli",
    8: "Agustus",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Desember"
}

monthly_pivot["bulan"] = monthly_pivot["bulan_index"].map(bulan_reverse_map)
monthly_pivot = monthly_pivot.sort_values("bulan_index")


####Preview

In [ ]:
monthly_pivot


In [ ]:
monthly_pivot[[
    "bulan",
    "total_pemasukan",
    "total_pengeluaran",
    "net_cash_flow"
]]


In [ ]:
df.sample(5)

Line Chart (Pemasukan vs pengeluaran)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(
    monthly_pivot["bulan"],
    monthly_pivot["total_pemasukan"],
    marker="o",
    label="Pemasukan"
)
plt.plot(
    monthly_pivot["bulan"],
    monthly_pivot["total_pengeluaran"],
    marker="o",
    label="Pengeluaran"
)

plt.title("Tren Pemasukan vs Pengeluaran Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah (Rp)")
plt.legend()
plt.grid(True)
plt.savefig("in vs out_white.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))

# Pemasukan → Hijau (positif)
plt.plot(
    monthly_pivot["bulan"],
    monthly_pivot["total_pemasukan"],
    marker="o",
    linewidth=2.5,
    color="#2ecc71",      # green
    label="Pemasukan"
)

# Pengeluaran → Oranye kemerahan (kontras)
plt.plot(
    monthly_pivot["bulan"],
    monthly_pivot["total_pengeluaran"],
    marker="o",
    linewidth=2.5,
    color="#e67e22",      # orange
    label="Pengeluaran"
)

# Judul & Label
plt.title(
    "Tren Pemasukan vs Pengeluaran Bulanan",
    fontsize=14,
    fontweight="bold",
    color="white"
)
plt.xlabel("Bulan", fontsize=11, color="white")
plt.ylabel("Jumlah (Rp)", fontsize=11, color="white")

# Grid halus
plt.grid(
    True,
    linestyle="--",
    alpha=0.3
)

# Legend dengan style
plt.legend(
    frameon=True,
    facecolor="#1e1e1e",
    edgecolor="white",
    fontsize=10
)
import matplotlib.ticker as ticker
plt.gca().yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"Rp{x/1e6:.1f}M")
)


plt.tight_layout()

plt.savefig(
    "in vs out",
    dpi=300,
    bbox_inches="tight",
    facecolor="#0E1117"
)
plt.show()


In [ ]:
df_monthly_cashflow = monthly_pivot[[
    "bulan_index",
    "bulan",
    "total_pemasukan",
    "total_pengeluaran",
    "net_cash_flow"
]].copy()


In [ ]:
df_monthly_cashflow.to_csv(
    "df_monthly_cashflow_dashboard.csv",
    index=False
)


In [ ]:
df_monthly_cashflow.info()
df_monthly_cashflow.head(10)


####Bar Chart Net Cash Flow

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(
    monthly_pivot["bulan"],
    monthly_pivot["net_cash_flow"]
)

plt.axhline(0)
plt.title("Net Cash Flow Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Surplus / Defisit (Rp)")
plt.grid(axis="y")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))

# Warna bar berdasarkan nilai net cash flow
colors = [
    "#2ecc71" if x >= 0 else "#e74c3c"
    for x in monthly_pivot["net_cash_flow"]
]

plt.bar(
    monthly_pivot["bulan"],
    monthly_pivot["net_cash_flow"],
    color=colors,
    width=0.6
)

# Garis nol (break-even line)
plt.axhline(
    0,
    color="white",
    linewidth=1,
    linestyle="--",
    alpha=0.8
)

# Judul & label
plt.title(
    "Net Cash Flow Bulanan",
    fontsize=14,
    fontweight="bold",
    color="white"
)
plt.xlabel("Bulan", fontsize=11, color="white")
plt.ylabel("Surplus / Defisit (Rp)", fontsize=11, color="white")

# Grid horizontal saja
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))

colors = [
    "#00E676" if x >= 0 else "#FF1744"
    for x in monthly_pivot["net_cash_flow"]
]

plt.bar(
    monthly_pivot["bulan"],
    monthly_pivot["net_cash_flow"],
    color=colors,
    width=0.6
)

plt.axhline(0, color="white", linestyle="--", linewidth=1, alpha=0.8)

plt.title("Net Cash Flow Bulanan", fontsize=14, fontweight="bold")
plt.xlabel("Bulan")
plt.ylabel("Surplus / Defisit (Rp)")
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(
    "net_cash_flow_bulanan.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="#0E1117"
)
plt.show()


###Sanity Check Final

In [ ]:
monthly_pivot.describe()


In [ ]:
monthly_pivot["net_cash_flow"].sum()


In [ ]:
monthly_pivot_stats = monthly_pivot.describe()


In [ ]:
monthly_pivot_stats.reset_index().to_csv(
    "monthly_cashflow_statistics.csv",
    index=False
)


##2. Analisa Struktur Pengeluaran

In [ ]:
import re
from collections import Counter

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # buang angka & simbol
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
all_words = []

for text in df["uraian"].dropna():
    cleaned = clean_text(text)
    words = cleaned.split()
    all_words.extend(words)

word_freq = Counter(all_words)


In [ ]:
pd.DataFrame(
    word_freq.most_common(100),
    columns=["keyword", "frequency"]
)

In [ ]:
stopwords = {
    "dan", "di", "ke", "dari", "yang",
    "pcs", "kg", "pak", "dus", "bal",
    "s", "sd", "tanggal",
    "rp", "idr"
}

In [ ]:
filtered_words = [
    w for w in all_words
    if w not in stopwords and len(w) > 2
]

filtered_freq = Counter(filtered_words)

pd.DataFrame(
    filtered_freq.most_common(30),
    columns=["keyword", "frequency"]
)

In [ ]:
out_words = []

for text in df[df["io_clean"] == "out"]["uraian"]:
    cleaned = clean_text(text)
    out_words.extend(cleaned.split())

out_freq = Counter(out_words)

pd.DataFrame(
    out_freq.most_common(20),
    columns=["keyword", "frequency"]
)


In [ ]:
in_words = []

for text in df[df["io_clean"] == "in"]["uraian"]:
    cleaned = clean_text(text)
    in_words.extend(cleaned.split())

in_freq = Counter(in_words)

pd.DataFrame(
    in_freq.most_common(20),
    columns=["keyword", "frequency"]
)


In [ ]:
keyword_df = pd.DataFrame(
    filtered_freq.most_common(),
    columns=["keyword", "frequency"]
)

keyword_df.to_csv("uraian_keyword_dictionary.csv", index=False)


In [ ]:
keyword_df.head(200)

####Filter data

In [ ]:
df_out = df[df["io_clean"] == "out"].copy()


In [ ]:
df_out.shape
df_out["amount"].sum()


#Mapping Kata Kunci

In [ ]:
def map_cost_category(uraian):
    if pd.isna(uraian):
        return "Lain-lain"

    u = uraian.lower()

    # =====================
    # 1️⃣ SDM (HIGHEST PRIORITY)
    # =====================
    if any(k in u for k in [
        "aldo", "gaji", "gaji aldo", "upah", "honor", "kasbon"
    ]):
        return "SDM"

    # =====================
    # 2️⃣ SEWA
    # =====================
    if any(k in u for k in ["sewa", "aplikasi"]):
        return "Sewa"

    # =====================
    # 3️⃣ UTILITIES
    # =====================
    if any(k in u for k in [
        "listrik", "air", "wifi", "internet", "token", "gas",
        "kantong", "admin", "panggonan"
    ]):
        return "Utilities"

    # =====================
    # 4️⃣ MARKETING / PLATFORM
    # =====================
    if any(k in u for k in [
        "grab", "gojek", "promo", "iklan",
        "tiktok","marketing", "merchant",
        "banner", "sticker", "cetak"
    ]):
        return "Marketing / Platform Fee"

    # =====================
    # 5️⃣ BAHAN BAKU (LOWER PRIORITY)
    # =====================
    if any(k in u for k in [
        "kopi", "coffee", "beans", "arabica", "robusta",
        "susu", "greenfield", "whip",
        "gula", "sugar", "palm",
        "sirup", "syrup", "caramel",
        "coklat", "matcha",
        "es", "le mineral",
        "ayam", "chicken", "beef", "bakso",
        "tahu", "telur", "morin", "peyek",
        "kentang", "beras", "roti",
        "dimsum", "cireng", "tomat",
        "saos", "sambal", "teriyaki",
        "indomie", "minyak", "monin",
        "kale", "cabe", "juice", "jungle juice",
        "delifru", "mamayo", "mayonaise", "telor",
        "bakemart", "nutrifarm", "rosela", "pakcoy",
        "outside", "ketimun", "uht diamond"
    ]):
        return "Bahan Baku"

    # =====================
    # 6️⃣ OPERASIONAL HARIAN
    # =====================
    if any(k in u for k in [
        "tisu", "cup", "sedotan", "kemasan", "plastik", "paper",
        "refund", "gelas", "fotokopi", "baterai",
        "transfer", "tukang", "iuran", "ambar",
        	"ambar (fotokopi)", "botol", "tissue", "tissu"
    ]):
        return "Operasional Harian"

    return "Lain-lain"


In [ ]:
df_out["cost_category"] = df_out["uraian_clean"].apply(map_cost_category)


##Klasifikasi pengeluaran tanpa membagi dataframe

In [ ]:
df["cost_category"] = df.apply(
    lambda row: map_cost_category(row["uraian_clean"])
    if row["io_clean"] == "out"
    else None,
    axis=1
)


In [ ]:
df.sample(5)

In [ ]:
df["cost_category"].value_counts()

###Analisa Komposisi Biaya

In [ ]:
cost_structure = (
    df_out
    .groupby("cost_category")["amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

cost_structure


In [ ]:
total_cost = cost_structure["amount"].sum()

cost_structure["percentage"] = (
    cost_structure["amount"] / total_cost * 100
).round(2)

cost_structure


In [ ]:
df_out.sample(5)

In [ ]:
uraian_by_category = (
    df_out[df_out["io_clean"] == "out"]
    .groupby(["cost_category", "uraian_clean"])
    .size()
    .reset_index(name="jumlah_transaksi")
    .sort_values(["cost_category", "jumlah_transaksi"], ascending=[True, False])
)


In [ ]:
uraian_by_category

In [ ]:
cost_category_dfs = {
    category: df_out[df_out["cost_category"] == category].copy()
    for category in df_out["cost_category"].unique()
}


In [ ]:
df_bahan_baku = cost_category_dfs["Bahan Baku"]
df_utilities = cost_category_dfs["Utilities"]


In [ ]:
cost_category_dfs["Bahan Baku"].head()


In [ ]:
for category, dfi in cost_category_dfs.items():
    filename = category.lower().replace(" ", "_").replace("/", "_")
    dfi.to_csv(f"cost_category_{filename}.csv", index=False)


###Struktur Pengeluaran

In [ ]:
import matplotlib.pyplot as plt

#plt.style.use("dark_background")

plt.figure(figsize=(11,6))

bars = plt.bar(
    cost_structure["cost_category"],
    cost_structure["amount"]
)

# Warna bar (teal / cyan kontras)
for bar in bars:
    bar.set_color("#00d4ff")   # cyan terang
    bar.set_edgecolor("white")
    bar.set_linewidth(0.8)

plt.title(
    "Struktur Pengeluaran Coffeeshop",
    fontsize=14,
    fontweight="bold",
    pad=15
)

max_idx = cost_structure["amount"].idxmax()
bars[max_idx].set_color("#ff4c4c")  # merah terang
plt.xlabel("Kategori Biaya", fontsize=11)
plt.ylabel("Total Pengeluaran (Rp)", fontsize=11)

plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    "cost_structure.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="#0E1117"
)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort data by amount descending
cost_structure_sorted = cost_structure.sort_values(
    "amount", ascending=False
).reset_index(drop=True)

plt.figure(figsize=(11,6))

bars = plt.bar(
    cost_structure_sorted["cost_category"],
    cost_structure_sorted["amount"]
)

# Palet warna berbasis risiko pengeluaran
color_palette = [
    "#ff2e2e",  # merah bold (tertinggi)
    "#ff9f1c",  # orange / amber
    "#ffd166",  # kuning hangat
    "#00d4ff",  # cyan
    "#4cc9f0",  # biru muda
    "#90dbf4"   # fallback jika kategori banyak
]

for i, bar in enumerate(bars):
    bar.set_color(color_palette[i] if i < len(color_palette) else "#90dbf4")
    bar.set_edgecolor("white")
    bar.set_linewidth(0.8)

plt.title(
    "Struktur Pengeluaran Coffeeshop",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.xlabel("Kategori Biaya", fontsize=11)
plt.ylabel("Total Pengeluaran (Rp)", fontsize=11)

plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(
    "cost_structure_sorted.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="#0E1117"
)
plt.show()


###Struktur Biaya per Bulan

In [ ]:
monthly_cost = (
    df_out
    .groupby(["bulan_index", "cost_category"])["amount"]
    .sum()
    .reset_index()
)

monthly_cost.head(10)


###Pivot untuk Eksplorasi

In [ ]:
monthly_cost_pivot = (
    monthly_cost
    .pivot(index="bulan_index", columns="cost_category", values="amount")
    .fillna(0)
)

monthly_cost_pivot


In [ ]:
monthly_cost_pivot.to_csv(
    "monthly_cost_pivot.csv",
    index=True
)


##3. Efisiensi Pengeluaran

###Filter Data

In [ ]:
df_eff = df[df["io_clean"] == "out"].copy()


###Fokus Ke Kategori Relevan

In [ ]:
def map_cost_category(uraian):
    if pd.isna(uraian):
        return "Lain-lain"

    u = uraian.lower()

    # =====================
    # 1️⃣ SDM (HIGHEST PRIORITY)
    # =====================
    if any(k in u for k in [
        "aldo", "gaji", "gaji aldo", "upah", "honor", "kasbon"
    ]):
        return "SDM"

    # =====================
    # 2️⃣ SEWA
    # =====================
    if any(k in u for k in ["sewa", "aplikasi"]):
        return "Sewa"

    # =====================
    # 3️⃣ UTILITIES
    # =====================
    if any(k in u for k in [
        "listrik", "air", "wifi", "internet", "token", "gas",
        "kantong", "admin", "panggonan"
    ]):
        return "Utilities"

    # =====================
    # 4️⃣ MARKETING / PLATFORM
    # =====================
    if any(k in u for k in [
        "grab", "gojek", "promo", "iklan",
        "tiktok","marketing", "merchant",
        "banner", "sticker", "cetak"
    ]):
        return "Marketing / Platform Fee"

    # =====================
    # 5️⃣ BAHAN BAKU (LOWER PRIORITY)
    # =====================
    if any(k in u for k in [
        "kopi", "coffee", "beans", "arabica", "robusta",
        "susu", "greenfield", "whip",
        "gula", "sugar", "palm",
        "sirup", "syrup", "caramel",
        "coklat", "matcha",
        "es", "le mineral",
        "ayam", "chicken", "beef", "bakso",
        "tahu", "telur", "morin", "peyek",
        "kentang", "beras", "roti",
        "dimsum", "cireng", "tomat",
        "saos", "sambal", "teriyaki",
        "indomie", "minyak", "monin",
        "kale", "cabe", "juice", "jungle juice",
        "delifru", "mamayo", "mayonaise", "telor",
        "bakemart", "nutrifarm", "rosela", "pakcoy",
        "outside", "ketimun", "uht diamond"
    ]):
        return "Bahan Baku"

    # =====================
    # 6️⃣ OPERASIONAL HARIAN
    # =====================
    if any(k in u for k in [
        "tisu", "cup", "sedotan", "kemasan", "plastik", "paper",
        "refund", "gelas", "fotokopi", "baterai",
        "transfer", "tukang", "iuran", "ambar",
        	"ambar (fotokopi)", "botol", "tissue", "tissu"
    ]):
        return "Operasional Harian"

    return "Lain-lain"


In [ ]:
df_eff["cost_category"] = df["uraian_clean"].astype(str).apply(map_cost_category)


In [ ]:
df_eff["cost_category"].value_counts()


In [ ]:
df_eff = df_eff[
    df_eff["cost_category"].isin([
        "Bahan Baku",
        "Operasional Harian"
    ])
]


In [ ]:
df_eff.shape


###qty_value

In [ ]:
df_eff[["qty", "qty_value", "qty_unit"]].head(10)


In [ ]:
df_eff = df_eff[df_eff["qty_value"] > 0]


#Menghitung Biaya per Unit

In [ ]:
df_eff["unit_cost"] = df_eff["amount"] / df_eff["qty_value"]


In [ ]:
df_eff.sort_values("unit_cost", ascending=False).head(10)


###Rata-rata Unit Cost per Uraian

In [ ]:
efficiency_item = (
    df_eff
    .groupby("uraian_clean")
    .agg(
        total_qty=("qty_value", "sum"),
        total_cost=("amount", "sum"),
        avg_unit_cost=("unit_cost", "mean"),
        transaksi=("amount", "count")
    )
    .reset_index()
    .sort_values("avg_unit_cost", ascending=False)
)


In [ ]:
efficiency_item.head(10)


In [ ]:
top10_transaksi = (
    efficiency_item
    .sort_values("transaksi", ascending=False)
    .head(10)
)


In [ ]:
top10_transaksi

In [ ]:
top10_highvaluetransaksi = (
    efficiency_item
    .sort_values(
        ["transaksi", "total_cost"],
        ascending=[False, False]
    )
    .head(10)
)
top10_highvaluetransaksi

###TREND Efisiensi per Bulan

In [ ]:
df_eff["qty_value"].value_counts()

In [ ]:
monthly_eff = (
    df_eff
    .groupby(["bulan_index"])
    .agg(
        total_cost=("amount", "sum"),
        total_qty=("qty_value", "sum")
    )
    .reset_index()
)

monthly_eff["avg_unit_cost"] = (
    monthly_eff["total_cost"] / monthly_eff["total_qty"]
)

monthly_eff


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(
    monthly_eff["bulan_index"],
    monthly_eff["avg_unit_cost"],
    marker="o"
)

plt.title("Tren Rata-rata Biaya per Unit")
plt.xlabel("Bulan")
plt.ylabel("Biaya per Unit (Rp)")
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11,6))

plt.plot(
    monthly_eff["bulan_index"],
    monthly_eff["avg_unit_cost"],
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="#6f4e37"  # coffee brown
)

# Highlight titik tertinggi
max_idx = monthly_eff["avg_unit_cost"].idxmax()
plt.scatter(
    monthly_eff.loc[max_idx, "bulan_index"],
    monthly_eff.loc[max_idx, "avg_unit_cost"],
    color="#c0392b",   # red accent
    s=90,
    zorder=3,
    label="Biaya Tertinggi"
)

plt.title(
    "Tren Rata-rata Biaya per Unit",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.xlabel("Bulan", fontsize=11)
plt.ylabel("Biaya per Unit (Rp)", fontsize=11)

plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11,6))

plt.plot(
    monthly_eff["bulan_index"],
    monthly_eff["avg_unit_cost"],
    linewidth=3,
    color="#f39c12",     # burnt orange (terang & kontras)
    zorder=2
)

plt.scatter(
    monthly_eff["bulan_index"],
    monthly_eff["avg_unit_cost"],
    s=70,
    color="#4e342e",     # coffee brown (kontras dg garis)
    edgecolor="white",
    linewidth=0.8,
    zorder=3
)

# Highlight titik tertinggi
max_idx = monthly_eff["avg_unit_cost"].idxmax()
plt.scatter(
    monthly_eff.loc[max_idx, "bulan_index"],
    monthly_eff.loc[max_idx, "avg_unit_cost"],
    s=120,
    color="#c0392b",     # red alert
    edgecolor="white",
    linewidth=1.2,
    zorder=4,
    label="Biaya Tertinggi"
)

plt.title(
    "Tren Rata-rata Biaya per Unit",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.xlabel("Bulan", fontsize=11)
plt.ylabel("Biaya per Unit (Rp)", fontsize=11)

plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()


##4. Pemasukan per Channel

####Mapping Chanel Pembayaran

In [ ]:
def map_channel(text):
    t = text.lower()

    if "grab" in t or "visionet" in t:
        return "Grab"

    elif "gojek" in t or "dompet anak bangsa" in t or "dompet anak bangsa (gojek)" in t:
        return "Gojek"

    elif "qr" in t:
        return "QRIS"

    elif "cash" in t:
        return "Cash"

    elif "cc" in t or "credit" in t:
        return "Credit Card"

    else:
        return "Other"


####Data Pemasukan

In [ ]:
df_in = df[df["io_clean"] == "in"].copy()

df_in["payment_channel"] = df_in["uraian"].astype(str).apply(map_channel)


In [ ]:
df_in["payment_channel"].value_counts()


In [ ]:
df_in.to_csv("df_in.csv", index=False)

In [ ]:
df.to_csv("final_analisa_cashflow.csv", index=False)

####Kontribusi Channel terhadap Total Revenue

In [ ]:
channel_revenue = (
    df_in
    .groupby("payment_channel")["amount"]
    .sum()
    .sort_values(ascending=False)
)
channel_revenue

####Proporsi

In [ ]:
channel_revenue_pct = (
    channel_revenue / channel_revenue.sum() * 100
).round(2)
channel_revenue_pct

In [ ]:
channel_revenue_pct.to_csv("channel_revenue_pct.csv")

In [ ]:
revenue_channel_summary = pd.DataFrame({
    "total_revenue": channel_revenue,
    "contribution_pct": channel_revenue_pct
})
revenue_channel_summary

In [ ]:
revenue_channel_summary.to_csv("revenue_channel_summary.csv")

####Tren Pemasukan per Channel

In [ ]:
channel_monthly = (
    df_in
    .groupby(["bulan", "payment_channel"])["amount"]
    .sum()
    .reset_index()
)


In [ ]:
channel_monthly_pivot = channel_monthly.pivot(
    index="bulan",
    columns="payment_channel",
    values="amount"
).fillna(0)
channel_monthly_pivot

####Stabilitas Channel

In [ ]:
channel_volatility = (
    channel_monthly
    .groupby("payment_channel")["amount"]
    .std()
    .sort_values(ascending=False)
)


In [ ]:
channel_stats = (
    channel_monthly
    .groupby("payment_channel")["amount"]
    .agg(["mean", "std"])
)

channel_stats["cv"] = channel_stats["std"] / channel_stats["mean"]
channel_stats.sort_values("cv")


In [ ]:
pd.set_option('display.max_columns', None)

others = df_in[df_in["payment_channel"] == "Other"]
others.head(100)

In [ ]:
df_in.sample(5)

In [ ]:
others[["uraian", "tanggal", "bulan", "amount"]] \
    .sort_values("amount", ascending=False) \
    .head(10)



In [ ]:
pd.set_option('display.max_columns', None)

others = df_in[df_in["payment_channel"] == "Other"]

top10_uraian = (
    others
    .groupby("uraian", as_index=False)["amount"]
    .sum()
    .sort_values("amount", ascending=False)
    .head(10)
)

top10_uraian


In [ ]:
pd.set_option('display.max_columns', None)

others = df_in[df_in["payment_channel"] == "Other"]
monthly_other = (
    others
    .groupby(["bulan_index", "bulan"], as_index=False)["amount"]
    .sum()
    .sort_values("bulan_index")
)


In [ ]:
monthly_other


####Klasifikasi nilai other pada kolom payment channel menjadi sewa vs non sewa

In [ ]:
others.head(100)

In [ ]:
def map_channel(text):
    s = text.lower()

    if "refund" in s or "rekening" in s or "deposit" in s or "transfer" in s or "customer" in s or "tf" in s or "tokopedia" in s:
        return "non sewa"

    else:
        return "sewa"

In [ ]:
others["sewa/nonsewa"] = others["uraian"].astype(str).apply(map_channel)

In [ ]:
others["sewa/nonsewa"].value_counts()

In [ ]:
others.sample(3)

In [ ]:
pd.set_option('display.max_columns', None)

others_nonsewa = others[others["sewa/nonsewa"] == "non sewa"]
monthly_other_nonsewa = (
    others_nonsewa
    .groupby(["bulan_index", "bulan"], as_index=False)["amount"]
    .sum()
    .sort_values("bulan_index")
)

In [ ]:
monthly_other_nonsewa

In [ ]:
pd.set_option('display.max_columns', None)

others_sewa = others[others["sewa/nonsewa"] == "sewa"]
monthly_other_sewa = (
    others_sewa
    .groupby(["bulan_index", "bulan"], as_index=False)["amount"]
    .sum()
    .sort_values("bulan_index")
)

In [ ]:
monthly_other_sewa

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9,4))

plt.plot(
    monthly_other_nonsewa["bulan"],
    monthly_other_nonsewa["amount"],
    marker="o",
    linewidth=2
)

plt.title("Tren Revenue Channel Other (Non Sewa)")
plt.xlabel("Bulan")
plt.ylabel("Total Revenue (Rp)")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9,4))

plt.plot(
    monthly_other_sewa["bulan"],
    monthly_other_sewa["amount"],
    marker="o",
    linewidth=2
)

plt.title("Tren Revenue Channel Other (Sewa)")
plt.xlabel("Bulan")
plt.ylabel("Total Revenue (Rp)")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###sewa vs non sewa

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(
    monthly_other_sewa["bulan"],
    monthly_other_sewa["amount"],
    marker="o",
    label="sewa"
)
plt.plot(
    monthly_other_nonsewa["bulan"],
    monthly_other_nonsewa["amount"],
    marker="o",
    label="non sewa"
)

plt.title("Trend Sewa & Non Sewa Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah (Rp)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
monthly_pivot

In [ ]:
monthly_other

In [ ]:
monthly_pivot2 = (
    monthly_pivot
    .merge(
        monthly_other[["bulan_index", "amount"]],
        on="bulan_index",
        how="left"
    )
    .rename(columns={"amount": "total_other"})
)


In [ ]:
monthly_pivot2

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(
    monthly_pivot2["bulan"],
    monthly_pivot2["total_other"],
    marker="o",
    label="Sewa / Nonsewa"
)
plt.plot(
    monthly_pivot2["bulan"],
    monthly_pivot2["total_pemasukan"],
    marker="o",
    label="Total Pemasukan"
)

plt.title("Trend Sewa/Nonsewa & Total Pemasukan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah (Rp)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
total_revenue = df_in["amount"].sum()


In [ ]:
other_revenue = df_in.loc[
    df_in["payment_channel"] == "Other", "amount"
].sum()


In [ ]:
other_share_pct = (other_revenue / total_revenue) * 100

total_revenue, other_revenue, other_share_pct


In [ ]:
monthly_total = (
    df_in
    .groupby(["bulan_index", "bulan"], as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "total_revenue"})
)


In [ ]:
monthly_other = (
    df_in[df_in["payment_channel"] == "Other"]
    .groupby(["bulan_index", "bulan"], as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "other_revenue"})
)


In [ ]:
monthly_share = (
    monthly_total
    .merge(monthly_other, on=["bulan_index", "bulan"], how="left")
    .fillna(0)
)

monthly_share["other_share_pct"] = (
    monthly_share["other_revenue"] / monthly_share["total_revenue"] * 100
)

monthly_share = monthly_share.sort_values("bulan_index")
monthly_share


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9,4))

plt.bar(
    monthly_share["bulan"],
    monthly_share["other_share_pct"]
)

plt.title("Kontribusi Revenue Channel Other per Bulan (%)")
plt.xlabel("Bulan")
plt.ylabel("Persentase (%)")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9,4))

bars = plt.bar(
    monthly_share["bulan"],
    monthly_share["other_share_pct"]
)

# Warna dasar: hijau emerald (opportunity / growth)
for bar in bars:
    bar.set_color("#00c853")      # hijau terang & bold
    bar.set_edgecolor("black")
    bar.set_linewidth(0.8)

# Highlight bulan dengan kontribusi tertinggi
max_idx = monthly_share["other_share_pct"].idxmax()
bars[max_idx].set_color("#ffb300")  # amber / emas (highlight opportunity)

plt.title(
    "Kontribusi Revenue Channel Other per Bulan (%)",
    fontsize=13,
    fontweight="bold",
    pad=12
)

plt.xlabel("Bulan", fontsize=11)
plt.ylabel("Persentase (%)", fontsize=11)

plt.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()


##5. Stabilitas & Resiko Bisnis

###Cashflow Volatility

In [ ]:
monthly_cf = (
    df
    .groupby(["bulan", "io_clean"])["amount"]
    .sum()
    .unstack(fill_value=0)
)

monthly_cf["net_cashflow"] = monthly_cf["in"] - monthly_cf["out"]


In [ ]:
monthly_cf

In [ ]:
cf_volatility = monthly_cf["net_cashflow"].std()
cf_mean = monthly_cf["net_cashflow"].mean()

cf_cv = cf_volatility / abs(cf_mean)
cf_cv


In [ ]:
cf_volatility

###Revenue Concentration Risk

In [ ]:
channel_share = (
    df_in
    .groupby("payment_channel")["amount"]
    .sum()
)

channel_pct = channel_share / channel_share.sum()


In [ ]:
channel_share

In [ ]:
channel_pct

In [ ]:
hhi = (channel_pct ** 2).sum()
hhi


###Cost Rigidity (Beban Tetap vs Fleksibel)

In [ ]:
fixed_costs = ["SDM", "Utilities", "Sewa"]


In [ ]:
df_out = df[df["io_clean"] == "out"].copy()

df_out["cost_category"] = df["uraian"].astype(str).apply(map_cost_category)

df_out["cost_type"] = df_out["cost_category"].apply(
    lambda x: "Fixed" if x in fixed_costs else "Variable"
)


####Proporsi beban tetap

In [ ]:
cost_structure = (
    df_out
    .groupby("cost_type")["amount"]
    .sum()
)

fixed_ratio = cost_structure["Fixed"] / cost_structure.sum()
fixed_ratio


###Operating Leverage (Sensitivitas Profit)

In [ ]:
operating_leverage_proxy = fixed_ratio / (1 - fixed_ratio)
operating_leverage_proxy


###Margin of Safety

In [ ]:
avg_revenue = monthly_cf["in"].mean()
avg_cost = monthly_cf["out"].mean()


In [ ]:
margin_of_safety = (avg_revenue - avg_cost) / avg_revenue
margin_of_safety


In [ ]:
# ============================================
# SIMPAN DATAFRAME KE FILE CSV
# ============================================

output_file = "analisa cashflow.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Dataframe berhasil disimpan ke file: {output_file}")

In [ ]:
df.sample(5)

In [ ]:
def map_cost_category(uraian):
    if pd.isna(uraian):
        return "Lain-lain"

    u = uraian.lower()

    # =====================
    # 1️⃣ SDM (HIGHEST PRIORITY)
    # =====================
    if any(k in u for k in [
        "aldo", "gaji", "gaji aldo", "upah", "honor", "kasbon"
    ]):
        return "SDM"

    # =====================
    # 2️⃣ SEWA
    # =====================
    if any(k in u for k in ["sewa", "aplikasi"]):
        return "Sewa"

    # =====================
    # 3️⃣ UTILITIES
    # =====================
    if any(k in u for k in [
        "listrik", "air", "wifi", "internet", "token", "gas",
        "kantong", "admin", "panggonan"
    ]):
        return "Utilities"

    # =====================
    # 4️⃣ MARKETING / PLATFORM
    # =====================
    if any(k in u for k in [
        "grab", "gojek", "promo", "iklan",
        "tiktok","marketing", "merchant",
        "banner", "sticker", "cetak"
    ]):
        return "Marketing / Platform Fee"

    # =====================
    # 5️⃣ BAHAN BAKU (LOWER PRIORITY)
    # =====================
    if any(k in u for k in [
        "kopi", "coffee", "beans", "arabica", "robusta",
        "susu", "greenfield", "whip",
        "gula", "sugar", "palm",
        "sirup", "syrup", "caramel",
        "coklat", "matcha",
        "es", "le mineral",
        "ayam", "chicken", "beef", "bakso",
        "tahu", "telur", "morin", "peyek",
        "kentang", "beras", "roti",
        "dimsum", "cireng", "tomat",
        "saos", "sambal", "teriyaki",
        "indomie", "minyak", "monin",
        "kale", "cabe", "juice", "jungle juice",
        "delifru", "mamayo", "mayonaise", "telor",
        "bakemart", "nutrifarm", "rosela", "pakcoy",
        "outside", "ketimun", "uht diamond"
    ]):
        return "Bahan Baku"

    # =====================
    # 6️⃣ OPERASIONAL HARIAN
    # =====================
    if any(k in u for k in [
        "tisu", "cup", "sedotan", "kemasan", "plastik", "paper",
        "refund", "gelas", "fotokopi", "baterai",
        "transfer", "tukang", "iuran", "ambar",
        	"ambar (fotokopi)", "botol", "tissue", "tissu"
    ]):
        return "Operasional Harian"

    return "Lain-lain"

In [ ]:
df["cost_category"] = df["uraian_clean"].apply(map_cost_category)

In [ ]:
df.sample(5)

##Analisa pengeluaran non bahan baku

In [ ]:
# Filter hanya pemasukan (IN)
df_revenue = df[df["i/o"] == "in"]

# Agregasi revenue bulanan
revenue_summary = (
    df_revenue
    .groupby("bulan_index")["cashflow_amount"]
    .sum()
    .sort_index()
)

# Plot
plt.figure()
revenue_summary.plot(kind="line", marker="o")
plt.title("Revenue Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Total Revenue")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# =========================
# 1. Revenue bulanan
# =========================
revenue_monthly = (
    df[df["i/o"] == "in"]
    .groupby("bulan_index")["cashflow_amount"]
    .sum()
)

expense_monthly = (
    df[
        (df["i/o"] == "out") &
        (df["cost_category"].isin(expense_categories))
    ]
    .groupby("bulan_index")["cashflow_amount"]
    .sum()
    .abs()
)


# =========================
# 3. Plot comparison
# =========================
plt.figure(figsize=(10, 6))

plt.plot(
    revenue_monthly.index,
    revenue_monthly.values,
    label="Revenue",
    linewidth=3,
    marker="o"
)

plt.plot(
    expense_monthly.index,
    expense_monthly.values,
    label="Total Expense",
    linewidth=3,
    marker="o"
)

plt.title("Perbandingan Revenue vs Expense Bulanan", fontsize=14, fontweight="bold")
plt.xlabel("Bulan")
plt.ylabel("Nilai (Rp)")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Revenue line
plt.plot(
    revenue_monthly.index,
    revenue_monthly.values,
    label="Revenue",
    linewidth=4,            # lebih tebal
    marker="o",
    markersize=8,
    color="#1B5E20"          # hijau gelap (kontras & profesional)
)

# Expense line
plt.plot(
    expense_monthly.index,
    expense_monthly.values,
    label="Total Expense",
    linewidth=4,            # lebih tebal
    marker="o",
    markersize=8,
    color="#B71C1C"          # merah gelap (expense)
)

plt.title(
    "Perbandingan Revenue vs Expense Bulanan",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Bulan", fontsize=11)
plt.ylabel("Nilai (Rp)", fontsize=11)

plt.legend(fontsize=11)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("revenue_expense_comparison.png", dpi=300)
plt.show()


###Analisa pengeluaran bahan baku - bahan baku untuk makanan dan minuman


In [ ]:
df_bahan_baku.head(10)

In [ ]:
def fnbcategory(uraian):
    if pd.isna(uraian):
        return "food"

    u = str(uraian).lower()

    # =====================
    # 1️⃣ Beverages
    # =====================
    if any(k in u for k in [
        "kopi", "coffee", "beans", "arabica", "robusta",
        "susu", "greenfield", "whip",
        "gula", "sugar", "palm",
        "sirup", "syrup", "caramel",
        "coklat", "matcha",
        "es", "le mineral", "juice", "jungle juice",
        "delifru", "uht", "diamond", "monin"
    ]):
        return "beverages"

    return "food"


In [ ]:
df_bahan_baku["food_beverages"] = df_bahan_baku["uraian_clean"].apply(fnbcategory)


In [ ]:
df_bahan_baku.sample(5)

##Total cost food vs beverages per bulan

In [ ]:
monthly_cost_fnb = (
    df_bahan_baku
    .groupby(["bulan", "food_beverages"], as_index=False)["amount"]
    .sum()
    .pivot(index="bulan", columns="food_beverages", values="amount")
    .fillna(0)
    .reset_index()
)

monthly_cost_fnb.to_csv("monthly_cost_fnb.csv", index=False)
monthly_cost_fnb


In [ ]:
monthly_cost_long = (
    df_bahan_baku
    .groupby(["bulan", "food_beverages"], as_index=False)["amount"]
    .sum()
)

monthly_cost_long


In [ ]:
import matplotlib.pyplot as plt

monthly = (
    df_bahan_baku
    .groupby(["bulan", "food_beverages"])["amount"]
    .sum()
    .unstack(fill_value=0)
)

plt.figure()

monthly.plot(
    kind="bar",
    color=["#E76F51", "#264653"] # food, beverages (kontras & profesional)
)

plt.title("Total Cost Food vs Beverages per Bulan")
plt.xlabel("Bulan")
plt.ylabel("Total Cost")
plt.xticks(rotation=45)

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

plt.savefig("total_cost_food_beverages.png", dpi=300)



In [ ]:
import matplotlib.pyplot as plt

yearly = (
    df_bahan_baku
    .groupby("food_beverages")["amount"]
    .sum()
)

plt.figure()

yearly.plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90,
    colors=["#F4A261", "#264653"]  # terang vs gelap (high contrast)
)

plt.title("Total Cost Food vs Beverages (Tahunan)")
plt.ylabel("")  # pie chart tidak perlu ylabel
plt.tight_layout()
plt.show()

plt.savefig("total_cost_food_beverages_yearly.png", dpi=300)

